In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
len(documents)

72

In [6]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [7]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [8]:
query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(
    query,
    num_results=5
)
results[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

In [9]:
results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [10]:
def search(query):
    return index.search(
        query,
        num_results=5
    )
results = search(
    "How does the agentic loop keep calling the model until it stops?"
)

results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [11]:
def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc["filename"])
        lines.append(doc["content"])
        lines.append("")
    return "\n".join(lines)

INSTRUCTIONS = """
You are a helpful course assistant.
Answer the question based on the provided context.
If the answer is not in the context, say "I don't know."
"""

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = f"""
    Question:
    {question}
    Context:
    {context}
    """
    return prompt

In [13]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()

def llm(prompt):
    response = client.responses.create(
        model="gpt-5.4-mini",
        input=[
            {
                "role":"developer",
                "content":INSTRUCTIONS
            },
            {
                "role":"user",
                "content":prompt
            }
        ]
    )
    return response

def rag(query):
    search_results = search(query)
    prompt = build_prompt(
        query,
        search_results
    )
    response = llm(prompt)
    return response

In [14]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(
        query,
        search_results
    )
    response = llm(prompt)
    return response
query = "How does the agentic loop keep calling the model until it stops?"
response = rag(query)

In [15]:
response.output_text

'It keeps calling the model in a `while True` loop. After each response, it checks whether the model returned any `function_call` items. If it did, the code runs the tool, appends the tool output to `messages`, and loops again. If there are no function calls on that turn, it breaks out of the loop and stops.'

In [16]:
response.usage.input_tokens

7101

In [17]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

In [18]:
len(chunks)

295

In [19]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [20]:
def search_chunk(query):
    return chunk_index.search(
        query,
        num_results=5
    )

results = search_chunk(
    "How does the agentic loop keep calling the model until it stops?"
)
results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [22]:
def rag_chunk(query):
    search_results = search_chunk(query)
    prompt = build_prompt(
        query,
        search_results
    )
    response = llm(prompt)

    return response

response = rag_chunk(
    "How does the agentic loop keep calling the model until it stops?"
)
response.usage.input_tokens

2284

In [23]:
def search(query: str) -> list[dict]:
    """
    Search the lesson documents for entries matching the given query.
    """
    return chunk_index.search(
        query,
        num_results=5
    )

from toyaikit.tools import Tools
agent_tools = Tools()
agent_tools.add_tool(search)

In [24]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the lesson documents for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [25]:
instructions = """
You're a course teaching assistant.
Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
""".strip()

from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(
        model="gpt-5.4-mini"
    )
)

result = runner.loop(
    prompt=
    "How does the agentic loop work, and how is it different from plain RAG?"
)

In [26]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nAnswer the student's question using the search tool.\nMake multiple searches with different keywords before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"agentic loop RAG difference"}', call_id='call_SSacmUv6MwrHItw19hQpVj9h', name='search', type='function_call', id='fc_0f515d03135c1193006a37e96ef3b88196ad1d48bdac5c4e6d', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"agentic loop"}', call_id='call_ePwAnIpFsbfKlMhCM51Hv4Xp', name='search', type='function_call', id='fc_0f515d03135c1193006a37e96ef3cc8196a7f39e5ce5618117', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"plain RAG retrieval augmented generation"}', call_id='call_quj4fEjfXLzXIOxA2Vd0KIyR', name='sea

In [32]:
count = 0

for item in result.all_messages:
    if getattr(item, "type", None) == "function_call":
        count += 1

print(count)

4
